In [7]:
# notebooks/test_transformers.ipynb
import sys
sys.path.append('..')
import pandas as pd
from src.transformers import DropColumnsTransformer

# Crear datos de prueba
df_test = pd.DataFrame({
    'id': [1, 2, 3],
    'nombre': ['A', 'B', 'C'],
    'edad': [25, 30, 35]
})

# Probar transformer
drop_cols = DropColumnsTransformer(columns_to_drop=['id', 'inexistente'])
df_resultado = drop_cols.fit_transform(df_test)

print("Original:", df_test.columns.tolist())
print("Resultado:", df_resultado.columns.tolist())
# Debería mostrar: ['nombre', 'edad']

[DropColumnsTransformer] Columnas eliminadas: ['id']
Original: ['id', 'nombre', 'edad']
Resultado: ['nombre', 'edad']


In [8]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from src.transformers import UnknownToNaNTransformer

# Datos de prueba
df_test = pd.DataFrame({
    'job': ['engineer', 'unknown', 'doctor', '', 'teacher', 'UNKNOWN'],
    'salary': [50000, 60000, 55000, 52000, 58000, 62000]
})

print("ANTES:")
print(df_test)
print(f"\nValores 'unknown' antes: {df_test['job'].isin(['unknown', '', 'UNKNOWN']).sum()}")

# Aplicar transformer
cleaner = UnknownToNaNTransformer()
df_clean = cleaner.fit_transform(df_test)

print("\nDESPUÉS:")
print(df_clean)
print(f"\nNaN después: {df_clean['job'].isna().sum()}")

ANTES:
        job  salary
0  engineer   50000
1   unknown   60000
2    doctor   55000
3             52000
4   teacher   58000
5   UNKNOWN   62000

Valores 'unknown' antes: 3
[UnknownToNaNTransformer] Columna 'job': 3 valores 'unknown' → NaN

DESPUÉS:
        job  salary
0  engineer   50000
1       NaN   60000
2    doctor   55000
3       NaN   52000
4   teacher   58000
5       NaN   62000

NaN después: 3


In [9]:
from src.transformers import DropHighMissingTransformer

# Crear DataFrame con diferentes niveles de nulos
df_test = pd.DataFrame({
    'col1': [1, 2, np.nan, 4, 5],           # 20% nulos
    'col2': [np.nan, np.nan, np.nan, 4, 5], # 60% nulos
    'col3': [np.nan, np.nan, np.nan, np.nan, np.nan]  # 100% nulos
})

print("Original:", df_test.columns.tolist())

drop_high = DropHighMissingTransformer(threshold=0.5)  # 50% umbral
drop_high.fit(df_test)
df_resultado = drop_high.transform(df_test)

print("Después:", df_resultado.columns.tolist())
# Debería mostrar: ['col1'] (col2 y col3 eliminadas)

Original: ['col1', 'col2', 'col3']
[DropHighMissingTransformer] Se eliminarán 2 columnas con >50.0% nulos:
   - col2: 60.0% nulos
   - col3: 100.0% nulos
Después: ['col1']


In [10]:
from src.transformers import DropZeroVarianceTransformer

df_test = pd.DataFrame({
    'normal': [1, 2, 3, 4, 5],
    'constante': [5, 5, 5, 5, 5],
    'casi_constante': [3, 3, 3, 3, 4]
})

drop_zero = DropZeroVarianceTransformer()
drop_zero.fit(df_test)
df_resultado = drop_zero.transform(df_test)

print("Original:", df_test.columns.tolist())
print("Después:", df_resultado.columns.tolist())
# Debería eliminar solo 'constante'

[DropZeroVariance] Columna 'constante' tiene varianza cero (todos los valores = 5) → será eliminada
Original: ['normal', 'constante', 'casi_constante']
Después: ['normal', 'casi_constante']


In [11]:
import sys
sys.path.append('..')
import pandas as pd
from src.transformers import OutlierCapper

# Tus datos de prueba
df_test = pd.DataFrame({
    'age': [25, 28, 30, 32, 1000, 27, 29, 31, 1500, 26],
    'normal': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
})

print("=" * 60)
print("PRUEBA CON IQR (MÉTODO ESTÁNDAR)")
print("=" * 60)

print("\nANTES - age:", df_test['age'].tolist())
print(f"Media original: {df_test['age'].mean():.2f}")

# Usar IQR con factor 1.5 (estándar)
capper = OutlierCapper(apply_capping=True, iqr_factor=1.5)
capper.fit(df_test)
df_resultado = capper.transform(df_test)

print("\nDESPUÉS - age:", df_resultado['age'].tolist())
print(f"Media después: {df_resultado['age'].mean():.2f}")

# Verificar outliers extremos
outliers_despues = [x for x in df_resultado['age'] if x > 100]
print(f"\n¿Quedaron valores > 100? {outliers_despues}")

PRUEBA CON IQR (MÉTODO ESTÁNDAR)

ANTES - age: [25, 28, 30, 32, 1000, 27, 29, 31, 1500, 26]
Media original: 272.80
[OutlierCapper] Columna 'age':
   - Q1 (25%): 27.25
   - Q3 (75%): 31.75
   - IQR: 4.50
   - Límite inferior: 20.50
   - Límite superior: 38.50
   - Outliers detectados: 2
   - Rango original: [25.00, 1500.00]
[OutlierCapper] Columna 'age':
   - Min: 25.00 → 25.00
   - Max: 1500.00 → 38.50

DESPUÉS - age: [25.0, 28.0, 30.0, 32.0, 38.5, 27.0, 29.0, 31.0, 38.5, 26.0]
Media después: 30.50

¿Quedaron valores > 100? []


In [12]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from src.transformers import SmartImputerTransformer

print("=" * 60)
print("PRUEBA COMPLETA DE SMART IMPUTER")
print("=" * 60)

# Datos de prueba con diferentes niveles de nulos
df_test = pd.DataFrame({
    'age': [25, np.nan, 30, 35, 28, 32, np.nan, 29, 31, 27],
    'job': ['engineer', 'doctor', np.nan, 'engineer', 'teacher', 
            'doctor', np.nan, 'engineer', 'teacher', 'doctor'],
    'salary': [50000, np.nan, 55000, np.nan, 52000, 
               np.nan, 48000, np.nan, 51000, 53000],
    'education': ['university', np.nan, 'high_school', np.nan, 'university',
                  np.nan, 'high_school', np.nan, 'university', 'master'],
})

print("\n1. DATOS ORIGINALES:")
print(df_test)
print(f"\nNulos totales: {df_test.isnull().sum().sum()}")

# Aplicar imputer
imputer = SmartImputerTransformer(low_threshold=0.10, high_threshold=0.50)
imputer.fit(df_test)
df_imputed = imputer.transform(df_test)

print("\n2. DATOS IMPUTADOS:")
print(df_imputed)
print(f"\nNulos después: {df_imputed.isnull().sum().sum()}")

# Verificaciones
assert df_imputed.isnull().sum().sum() == 0, "❌ Quedaron nulos sin imputar"
print("\n✅ SmartImputerTransformer funciona correctamente")
print("   - No quedan nulos en los datos")

PRUEBA COMPLETA DE SMART IMPUTER

1. DATOS ORIGINALES:
    age       job   salary    education
0  25.0  engineer  50000.0   university
1   NaN    doctor      NaN          NaN
2  30.0       NaN  55000.0  high_school
3  35.0  engineer      NaN          NaN
4  28.0   teacher  52000.0   university
5  32.0    doctor      NaN          NaN
6   NaN       NaN  48000.0  high_school
7  29.0  engineer      NaN          NaN
8  31.0   teacher  51000.0   university
9  27.0    doctor  53000.0       master

Nulos totales: 12
[SmartImputer] Columna 'age': 20.0% nulos → imputar con mediana=29.50
[SmartImputer] Columna 'job': 20.0% nulos → crear categoría 'missing'
[SmartImputer] Columna 'salary': 40.0% nulos → imputar con mediana=51500.00
[SmartImputer] Columna 'education': 40.0% nulos → crear categoría 'missing'
[SmartImputer] Columna 'age': imputados 2 nulos
[SmartImputer] Columna 'job': imputados 2 nulos
[SmartImputer] Columna 'salary': imputados 4 nulos
[SmartImputer] Columna 'education': imputados 4